# Exploratory Data Analysis
<!-- 
## Time spent
|  Name  | Time   |
| ---    |  ---   |
| Danylo | 40 min |
| Ivan   | 5 min  |
| Daryna | 0 min  |
-->
**Course**: Computational Social Science, Homework #2  
**Team**: FightClub  
**Research topic**: Brainrot analysis  
**Lecturer**: Andrew Kurochkin  
**Date**: October 2026  
**Place**: National University of Kyiv-Mohyla Academy, Faculty of Informatics

## Table of contents
0. [Overview](#Overview)
    1. Hypotheses
    2. Datasets
1. H1: Discourse is getting shorter and simper
    1. Overview
    2. Activity and distribution
    3. Content analysis
    4. Interesting findings
2. H3: Wellbeing is getting worse among doomscrollers
3. H4: Crises intensify doomscrolling and negativity
4. Result

## 0. Overview
Since our goal is many-sided and has certain specific characteristics, we will follow our own flow with exploration and analysis of data. We already have hypotheses to test and we will try to find approaches in order to prove those hypothese right or wrong. We will not follow guidelines or try to check any boxes, instead we will follow our curiosity and play until we have the answer.

However, there is one guideline: before freestyling with analysis - explore the data and brainstorm what visualizations should be made.

Since we have 4 different hypotheses to test, the research is structured into 4 independent parts, each of them testing one hypothesis and doing separate data overview. Each hypothesis is tested with separate datasets and those datasets are reviewed independently in each of 4 parts. After that some of the results and findings are merged.

### Hypotheses
Our topic is analysis of impact of doomscrolling and brainrot on people, discourse, their wellbeing and cognitive abilities over time. We study the effects of doomscrilling with the hypothese below.

- H1: Discourse is getting shorter and simpler
    - Lexical & NLP approach
    - Temporal/Behavioral aspect
- H2: Short-form platforms accelerated the degradation of discourse
- H3: Wellbeing is getting worse among doomscrollers
- H4: Crises intensify doomscrolling and negativity

### Datasets
Each hypothesis is proven by using different datasets.

#### H1: Discourse is getting shorter and simper
Commets and submissions from reddit, collected from 01.2012 to 09.2026, divided into different categories.

1. Short-form: r/memes, r/teenagers
2. Long-form: r/books, r/explainlikeimfive
3. Baseline: r/todayilearned
4. Qualitative: r/nosurf

#### H2: Short-form platforms accelerated the degradation of discourse
1. [Reels/Shorts Consumption vs Attention Span](https://www.kaggle.com/datasets/jayjoshi37/reelsshorts-consumption-vs-attention-span)

#### H3: Wellbeing is getting worse among doomscrollers
1. [Social Media Addiction dataset](https://www.kaggle.com/datasets/engieid/social-media-addiction-dataset)
2. [The Dark At The End Of The Tunnel: Doomscrolling On Social Media Newsfeeds](https://osf.io/u9f6h/)
3. [Sleep & Doomscrolling Habits Dataset](https://www.kaggle.com/datasets/harpartapsingh13/sleep-and-doomscrolling-habits-dataset)
4. [Social Media and Mental Health](https://www.kaggle.com/datasets/souvikahmed071/social-media-and-mental-health)
5. [Student Social Media & Mental Health](https://www.kaggle.com/datasets/shivasingh4945/student-social-media-and-mental-health-impact)

#### H4: Crises intensify doomscrolling and negativity


### Additional approches
- questionnaire
- YouTube comments

## 1. H1: Discourse is getting shorter and simpler
> Only Danylo works below
Time spent: 60min

In this section we test this hypothesis with data. We'll be analyzing the discourse of a dataset of 9.2 million comments and 1.8 million submissions that were posted on reddit between 2012 and 2026.

Before getting into data and into analysis, in order to pick the correct analysis path, let's take a bird's-eye view of our data and on our goal. We want to verify that over the 14 years discourse has been getting shorter and simpler. In order to check that, we have two approaches:
1. **Lexical & NLP approach**
2. **Temporal/Behavioral aspect**

In **Lexical & NLP approach** we analyse sentences and check the following metrics:
1. Vocabulary is shrinking
2. Sentences are getting shorter
    1. Words per comment
    2. Words per sentence
3. Lempel-Ziv algorithm
    1. Concatenation of comments into fixed-size chunks per subreddit per month

In **Temporal/Behavioral aspect** we analyze conversantions and look into:
1. Thread depth
    1. Thread size
    2. Share or replies vs top-level comments
    3. True depth <!-- download full month for that -->

Ideas for analysis:
1. Readability (Flesch-Kincaid)
2. Avg word length

### 1.1. Overview
Our dataset consists of 9,237,205 comments and 1,870,257 posts, collected from 01.2012 to 09.2026, from these subreddits:
1. Short-form: r/memes, r/teenagers
2. Long-form: r/books, r/explainlikeimfive
3. Baseline: r/todayilearned
4. Qualitative: r/nosurf

Dataset was downloaded from [Arctic Shift API](https://arctic-shift.photon-reddit.com/). We didn't download full dataset of comments and posts because of the size and downloading time. Instead, we sampled 10,000 comments per subreddit per month and 2,000 posts per subreddit per month. Samples are downloaded from random time slots. Each month is cut into equal time slots, each slot holding roughly 1,000 comments or 200 posts. For our subreddits total amount of comments is 346.4 million and total amount of posts is 27.3 million. Our sample covers about 2.7% of all comments and 6.9% of all posts.


In [1]:
"""
!Important!
Libraries needed:
 - 


Overview here:
- number of records
- timeframe
"""

"""
1. Setup
"""
from pathlib import Path
import duckdb
import pandas as pd

DATA = Path("data/reddit_arctic")
con = duckdb.connect()
con.sql("SET TimeZone = 'UTC'")   # Reddit timestamps are UTC; without this DuckDB uses your computer's time zone

FOLDERS = {"comments": "comments", "posts": "submissions"}
for view, folder in FOLDERS.items():
    con.sql(f"""CREATE OR REPLACE VIEW {view} AS
                SELECT * FROM read_parquet('{(DATA / folder).as_posix()}/**/*.parquet', hive_partitioning=false)""")

In [2]:
"""
2. Number of records, timeframe, size
"""
rows = []
for view, folder in FOLDERS.items():
    files = list((DATA / folder).rglob("*.parquet"))
    stats = con.sql(f"""
        SELECT count(*)                                                     AS records,
               count(DISTINCT subreddit)                                    AS subreddits,
               min(to_timestamp(created_utc))::DATE                         AS first_record,
               max(to_timestamp(created_utc))::DATE                         AS last_record,
               count(DISTINCT strftime(to_timestamp(created_utc), '%Y-%m')) AS months
        FROM {view}""").df().iloc[0].to_dict()
    rows.append({"type": view, **stats, "files": len(files),
                 "size_MB": round(sum(f.stat().st_size for f in files) / 1024**2, 1)})
overview = pd.DataFrame(rows).set_index("type")
overview

,records,subreddits,first_record,last_record,months,files,size_MB
type,,,,,,,
comments,9237205,6,2012-01-01,2026-09-30,177,1062,897.5
posts,1870257,6,2012-01-01,2026-09-30,177,1062,225.4


In [3]:
"""
3. Per subreddit, the sample vs everything on Reddit
"""
totals = pd.read_csv(DATA / "_monthly_totals.csv")
per_sub = []
for view, total_col in [("comments", "comments_total"), ("posts", "posts_total")]:
    s = con.sql(f"""SELECT subreddit, strftime(to_timestamp(created_utc), '%Y-%m') AS month, count(*) AS sampled
                    FROM {view} GROUP BY ALL""").df()
    m = totals[["subreddit", "month", total_col]].merge(s, on=["subreddit", "month"], how="left").fillna({"sampled": 0})
    g = m.groupby("subreddit").agg(sampled=("sampled", "sum"), total=(total_col, "sum"),
                                   first_month=("month", "min"), last_month=("month", "max"))
    g["coverage_%"] = (100 * g["sampled"] / g["total"]).round(1)
    g["complete_months"] = (m["sampled"] >= 0.99 * m[total_col]).groupby(m["subreddit"]).sum()
    g.columns = pd.MultiIndex.from_product([[view], g.columns])
    per_sub.append(g)
per_sub = pd.concat(per_sub, axis=1)
per_sub

comments                                               \
                   sampled      total first_month last_month coverage_%   
subreddit                                                                 
books              1876751   13137631     2012-01    2026-09       14.3   
explainlikeimfive  1852536   21058471     2012-01    2026-09        8.8   
memes              1334537  108528872     2012-01    2026-09        1.2   
nosurf              478540     478745     2012-01    2026-09      100.0   
teenagers          1843775  133272831     2012-01    2026-09        1.4   
todayilearned      1851066   69926671     2012-01    2026-09        2.6   

                                      posts                                   \
                  complete_months   sampled     total first_month last_month   
subreddit                                                                      
books                           0  368255.0    890216     2012-01    2026-09   
explainlikeimfive               0  369085.0   2000253     2012-01    2026-09   
memes                          62  343402.0  11858295     2012-01    2026-09   
nosurf                        177   56057.0     56082     2012-01    2026-09   
teenagers                       0  363975.0  10305407     2012-01    2026-09   
todayilearned                   0  369483.0   2162300     2012-01    2026-09   

                                              
                  coverage_% complete_months  
subreddit                                     
books                   41.4               0  
explainlikeimfive       18.5               0  
memes                    2.9              38  
nosurf                 100.0             177  
teenagers                3.5               8  
todayilearned           17.1               0

### 1.2. Activity and distribution
counts by month, subreddit

### 1.3. Content analysis
frequent terms, topics, or sentiment (if relevant).

### 1.4. Interesting findings
correlations, spikes, or patterns worth explaining

## H3 probably
> Only Darka works below
Time spent: 5 min

## H4 probably
> Only Ivan works below
Time spent: 0 min